<a href="https://colab.research.google.com/github/LefieTsimplekas/thesis-bikeshare-synthesis/blob/main/Notebooks/colab_train_tabddpm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Εκπαίδευση TabDDPM στο Google Colab

**Πριν ξεκινήσεις:** `Runtime → Change runtime type → T4 GPU`.

Στο Google Drive χρειάζεται ο φάκελος `MyDrive/thesis/model_input/` με:
- τα 9 αρχεία του `data/model_input/` (`A_train.parquet` … `D_val.parquet`, `london_test.parquet`)
- το `artifacts/model_input/schema.json`

Τα αποτελέσματα γράφονται στο `MyDrive/thesis/outputs/`, ώστε να μη χάνονται όταν κλείσει η συνεδρία.

## 1. Έλεγχος GPU

In [ ]:
!nvidia-smi

Tue Nov 10 18:56:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Εγκατάσταση (μία φορά ανά συνεδρία, ~3–5 λεπτά)

Το Colab τρέχει πλέον Python 3.13, για την οποία δεν υπάρχουν τα torch 2.2.2 και numpy 1.26.4 που απαιτεί το synthcity. Γι' αυτό φτιάχνεται ένα ξεχωριστό περιβάλλον **Python 3.11** με το `uv`, με τις ίδιες εκδόσεις όπως το `requirements.txt`. Τα scripts τρέχουν με αυτό (`/content/py311/bin/python`). **Δεν χρειάζεται Restart session.**

In [1]:
!pip install -q uv
!uv venv /content/py311 --python 3.11 -q
!uv pip install -q --python /content/py311/bin/python torch==2.2.2 numpy==1.26.4 pandas==2.3.3 pyarrow==24.0.0 scipy==1.17.1 scikit-learn==1.8.0 xgboost==2.1.4 synthcity==0.2.12 opacus==1.4.0

## 3. Drive, κώδικας και φάκελοι

In [2]:
cfrom google.colab import drive
drive.mount('/content/drive')

import os, subprocess
REPO = '/content/thesis-bikeshare-synthesis'
DRIVE = '/content/drive/MyDrive/thesis'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', 'https://github.com/LefieTsimplekas/thesis-bikeshare-synthesis', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull'], check=True)
os.chdir(REPO)

# Είσοδοι από το Drive
os.makedirs('data', exist_ok=True); os.makedirs('artifacts/model_input', exist_ok=True)
if not os.path.exists('data/model_input'):
    os.symlink(f'{DRIVE}/model_input', 'data/model_input')
if not os.path.exists('artifacts/model_input/schema.json'):
    os.symlink(f'{DRIVE}/model_input/schema.json', 'artifacts/model_input/schema.json')

# Έξοδοι στο Drive
for local, remote in [('data/synthetic', 'synthetic'), ('models', 'models'), ('artifacts/training', 'training')]:
    os.makedirs(f'{DRIVE}/outputs/{remote}', exist_ok=True)
    if not os.path.exists(local):
        os.symlink(f'{DRIVE}/outputs/{remote}', local)

print(sorted(os.listdir('data/model_input')))

PY = '/content/py311/bin/python'
subprocess.run([PY, '-c', "import sys, torch, numpy; print('python', sys.version.split()[0], '| torch', torch.__version__, '| CUDA', torch.cuda.is_available(), '| numpy', numpy.__version__)"], check=True)

Mounted at /content/drive
['manifest.json']


CompletedProcess(args=['/content/py311/bin/python', '-c', "import sys, torch, numpy; print('python', sys.version.split()[0], '| torch', torch.__version__, '| CUDA', torch.cuda.is_available(), '| numpy', numpy.__version__)"], returncode=0)

In [3]:
import os
os.chdir('/content/thesis-bikeshare-synthesis')
DRIVE = '/content/drive/MyDrive/thesis'
os.makedirs('data/model_input', exist_ok=True); os.makedirs('artifacts/model_input', exist_ok=True)
for f in os.listdir(f'{DRIVE}/model_input'):
    dst = 'artifacts/model_input/schema.json' if f == 'schema.json' else f'data/model_input/{f}'
    if not os.path.exists(dst):
        os.symlink(f'{DRIVE}/model_input/{f}', dst)
print(sorted(os.listdir('data/model_input')))

['A_train.parquet', 'A_val.parquet', 'B_train.parquet', 'B_val.parquet', 'C_train.parquet', 'C_val.parquet', 'D_train.parquet', 'D_val.parquet', 'london_test.parquet', 'manifest.json']


## 4. Πιλοτική εκτέλεση (σενάριο D, 50.000 γραμμές, 5.000 βήματα)

Ελέγχει ότι όλα δουλεύουν και μετρά την ταχύτητα. Στο «Quick check» στο τέλος, τα ποσοστά και οι διάμεσοι του συνθετικού πρέπει να είναι κοντά σε αυτούς του train.

In [ ]:
!/content/py311/bin/python src/07_train_tabddpm.py --scenario D --pilot-rows 50000 --steps 5000 --gen-rows 20000 2>&1 | grep -v 'Epoch'

[transformers] Disabling PyTorch because PyTorch >= 2.6 is required but found 2.2.2
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
[2026-11-09T21:56:03.232780+0000][5575][CRITICAL] Error importing TabularGoggle: No module named 'dgl'
[2026-11-09T21:56:03.235387+0000][5575][CRITICAL] module disabled: /content/py311/lib/python3.11/site-packages/synthcity/plugins/generic/plugin_goggle.py

===== TabDDPM | scenario D =====
Rows 50,000 | batch 4096 | 13 steps/epoch | 385 epochs ≈ 5,005 steps | device cuda
Training time 6.2 min (13.5 steps/s)
Generation of 20,000 rows: 0.9 min
Quick check (train vs synthetic):
  train     e-bike  18.2% | round trip  2.5% | median duration 13.34 min | median distance  2.66 km
  synthetic e-bike  18.1% | round trip  2.3% | median duration 13.52 min | median distance  2.71 km
Saved data/synthetic/tabddpm_D_pilot.parquet


## 5. Πλήρης εκπαίδευση (ένα σενάριο ανά κελί)

Τρέξε τα κελιά ένα ένα. Αν η συνεδρία κλείσει, ό,τι έχει τελειώσει είναι ήδη στο Drive· ξανατρέχεις τα κελιά 2 και 3 και μετά το επόμενο σενάριο.

In [ ]:
!/content/py311/bin/python src/07_train_tabddpm.py --scenario D 2>&1 | grep -v 'Epoch'

In [ ]:
!/content/py311/bin/python src/07_train_tabddpm.py --scenario A 2>&1 | grep -v 'Epoch'

In [ ]:
!/content/py311/bin/python src/07_train_tabddpm.py --scenario B 2>&1 | grep -v 'Epoch'

In [ ]:
!/content/py311/bin/python src/07_train_tabddpm.py --scenario C 2>&1 | grep -v 'Epoch'

## 6. Seeds: 3 training × 3 generation seeds (§7), με αυτόματη συνέχεια

**Runtime: T4 GPU.** Κάθε φορά που ανοίγεις νέα συνεδρία: κελιά **4 → 6 → 7**, μετά **6α** και **6β**.

- **6α** δείχνει τι υπάρχει ήδη στο Drive και πόσος χρόνος GPU μένει. Μπορείς να το τρέξεις όποτε θες.
- **6β** τρέχει **μόνο ό,τι λείπει** για τα σενάρια της λίστας `SCENARIOS`:
  - μοντέλο αποθηκευμένο → μόνο οι γενέσεις που λείπουν (`--generate-only`)
  - αλλιώς → εκπαίδευση και οι γενέσεις που λείπουν
  - αρχείο γένεσης που έμεινε μισό από διακοπή → ξαναγίνεται

  Αν η συνεδρία κοπεί, ανοίγεις νέα και ξανατρέχεις τα ίδια κελιά: συνεχίζει από εκεί που σταμάτησε. Χάνεται το πολύ μία εκπαίδευση στη μέση (~27 λεπτά).
- Με `AUTO_DISCONNECT = True`, στο τέλος του 6β γίνεται συγχρονισμός του Drive και **αποσύνδεση του runtime**, ώστε να μην καίγονται compute units αν δεν είσαι μπροστά. Η έξοδος του κελιού μένει ορατή. Αν σταματήσεις το κελί με ■, δεν γίνεται αποσύνδεση.
- **Η καρτέλα πρέπει να μείνει ανοιχτή** όσο τρέχει (χωρίς Pro+ δεν υπάρχει εκτέλεση στο παρασκήνιο).

Χρόνοι σε T4: εκπαίδευση ~27 λεπτά, γένεση 200k ~9 λεπτά. Ένα σενάριο από την αρχή ≈ 2 ώρες 15 λεπτά. Προτεινόμενη σειρά: A, C, B, D.

### 6α. Κατάσταση

In [4]:
import os, subprocess, time
import pyarrow.parquet as pq

os.chdir('/content/thesis-bikeshare-synthesis')
PY = '/content/py311/bin/python'
TRAIN_SEEDS = [42, 43, 44]
GEN_SEEDS = [1, 2, 3]
MIN_TRAIN, MIN_GEN = 27, 9            # λεπτά σε T4
N_ROWS = pq.read_metadata('data/model_input/london_test.parquet').num_rows   # 200.000

def model_path(sc, seed):
    p = f'models/tabddpm_{sc}_s{seed}.pkl'
    if os.path.exists(p):
        return p
    legacy = f'models/tabddpm_{sc}.pkl'          # πρώτη εκτέλεση· ισχύει μόνο για το seed 42
    return legacy if seed == 42 and os.path.exists(legacy) else None

def gen_ok(sc, seed, g):
    try:   # λείπει ή έμεινε μισό από διακοπή -> False
        return pq.read_metadata(f'data/synthetic/tabddpm_{sc}_s{seed}_g{g}.parquet').num_rows == N_ROWS
    except Exception:
        return False

def todo(sc, seed):
    missing = [g for g in GEN_SEEDS if not gen_ok(sc, seed, g)]
    train = bool(missing) and model_path(sc, seed) is None
    return missing, train

def status(scenarios=('A', 'C', 'B', 'D')):
    done_all, minutes = 0, 0
    for sc in scenarios:
        for seed in TRAIN_SEEDS:
            missing, train = todo(sc, seed)
            done = [g for g in GEN_SEEDS if g not in missing]
            done_all += len(done)
            minutes += MIN_TRAIN * train + MIN_GEN * len(missing)
            model = 'μοντέλο ✓' if model_path(sc, seed) else 'μοντέλο –'
            print(f'{sc}  s{seed}  {model}   γενέσεις έτοιμες: {done if done else "–"}')
        print()
    print(f'Έτοιμα: {done_all}/{9 * len(scenarios)} αρχεία · απομένουν ~{minutes / 60:.1f} ώρες GPU')

status()

A  s42  μοντέλο ✓   γενέσεις έτοιμες: [1, 2, 3]
A  s43  μοντέλο ✓   γενέσεις έτοιμες: [1, 2, 3]
A  s44  μοντέλο ✓   γενέσεις έτοιμες: [1, 2, 3]

C  s42  μοντέλο ✓   γενέσεις έτοιμες: –
C  s43  μοντέλο –   γενέσεις έτοιμες: –
C  s44  μοντέλο –   γενέσεις έτοιμες: –

B  s42  μοντέλο ✓   γενέσεις έτοιμες: –
B  s43  μοντέλο –   γενέσεις έτοιμες: –
B  s44  μοντέλο –   γενέσεις έτοιμες: –

D  s42  μοντέλο ✓   γενέσεις έτοιμες: –
D  s43  μοντέλο –   γενέσεις έτοιμες: –
D  s44  μοντέλο –   γενέσεις έτοιμες: –

Έτοιμα: 9/36 αρχεία · απομένουν ~6.8 ώρες GPU


### 6β. Εκτέλεση ό,τι λείπει

In [5]:
SCENARIOS = ['C']          # π.χ. ['A', 'C'] για δύο σενάρια στη σειρά (~4,5 ώρες)
AUTO_DISCONNECT = True     # αποσύνδεση στο τέλος, για να μην καίγονται compute units

def run(cmd):
    print('\n>>', ' '.join(cmd[1:]), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in p.stdout:
            if 'Epoch' not in line:
                print(line, end='', flush=True)
    except KeyboardInterrupt:
        p.kill()
        raise
    if p.wait() != 0:
        raise RuntimeError(f'Απέτυχε (κωδικός {p.returncode}): {" ".join(cmd[1:])}')

t0, stopped_by_user = time.time(), False
try:
    for sc in SCENARIOS:
        for seed in TRAIN_SEEDS:
            missing, train = todo(sc, seed)
            if not missing:
                print(f'{sc} s{seed}: έτοιμο, παραλείπεται')
                continue
            cmd = [PY, 'src/07_train_tabddpm.py', '--scenario', sc, '--seed', str(seed),
                   '--gen-seeds', *map(str, missing)]
            if not train:
                cmd.append('--generate-only')
            run(cmd)
    print(f'\nΤέλος σε {(time.time() - t0) / 60:.0f} λεπτά\n')
    status(SCENARIOS)
except KeyboardInterrupt:
    stopped_by_user = True
    print('\nΣταμάτησε από εσένα· το runtime μένει συνδεδεμένο.')
finally:
    if AUTO_DISCONNECT and not stopped_by_user:
        from google.colab import drive, runtime
        print('Συγχρονισμός Drive και αποσύνδεση runtime...', flush=True)
        drive.flush_and_unmount()
        runtime.unassign()


>> src/07_train_tabddpm.py --scenario C --seed 42 --gen-seeds 1 2 3 --generate-only
[transformers] Disabling PyTorch because PyTorch >= 2.6 is required but found 2.2.2
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
[KeOps] Compiling cuda jit compiler engine ... OK
[pyKeOps] Compiling nvrtc binder for python ... OK

===== TabDDPM | scenario C | training seed 42 =====
Loading models/tabddpm_C.pkl
Generation seed 1: 200,000 rows in 8.7 min -> data/synthetic/tabddpm_C_s42_g1.parquet
  synthetic e-bike  58.7% | round trip  3.1% | median duration  9.30 min | median distance  2.04 km
Generation seed 2: 200,000 rows in 8.7 min -> data/synthetic/tabddpm_C_s42_g2.parquet
  synthetic e-bike  58.5% | round trip  3.1% | median duration  9.30 min | median distance  2.04 km
Generation seed 3: 200,000 rows in 8.6 min -> data/synthetic/tabddpm_C_s42_g3.parquet
  synthetic e-bike  58.6% | round trip  3.1% | median 